In [ ]:
from pathlib import Path

from cluster1.data.kernels import get_kernel_spec
from cluster1.data.prompts.prompt_contract import build_prompt
from cluster1.generation.grammar_loader import load_compiled_grammar
from cluster1.constraints.hardware_checker import HardwareChecker
from cluster1.generation.constrained_gen import generate_source
from cluster1.validation.compile_check import check_compiles_all_dtypes
from cluster1.results.dataclass import GenerationResult, validate_result_invariants

MODEL_ID = "Qwen/Qwen2.5-Coder-7B-Instruct-AWQ"
DATASET_ID = "ScalingIntelligence/KernelBench"
GRAMMAR_PATH = Path("cluster1/grammar/triton_kernel.gbnf")
if not GRAMMAR_PATH.exists():
    GRAMMAR_PATH = Path("../grammar/triton_kernel.gbnf")
MODEL_NOTE = "Development iteration model for validation runs; not the final thesis reporting model."

MODEL_ID, DATASET_ID, MODEL_NOTE


In [ ]:
spec = get_kernel_spec("elementwise")
dtype = "fp32"
prompt = build_prompt(spec, dtype)

prompt_components = {
    "dataset_id": spec.dataset_id,
    "kernel_class": spec.kernel_class,
    "kernel_name": spec.name,
    "function_signature": f"{spec.launcher_name}{spec.reference_signature}",
    "dtype_device": f"{dtype} / CUDA",
    "fixed_autotune_configs": spec.autotune_configs,
    "prompt": prompt,
}
prompt_components


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID)
if hasattr(model, "eval"):
    model.eval()

compiled_grammar = load_compiled_grammar(str(GRAMMAR_PATH), MODEL_ID)
hardware_checker = HardwareChecker()
decoded = generate_source(
    prompt=prompt,
    model=model,
    tokenizer=tokenizer,
    grammar_active=True,
    compiled_grammar=compiled_grammar,
    hardware_checker=hardware_checker,
    max_new_tokens=1024,
    temperature=0.2,
    seed=0,
)

print(decoded.source)
{"masked_token_rate": decoded.masked_token_rate}


In [ ]:
import json
from dataclasses import asdict
from datetime import UTC, datetime
from uuid import uuid4

from cluster1.results.dataclass import compute_unique_solution_hash

compile_results = check_compiles_all_dtypes(
    decoded.source,
    spec.compile_spec,
    spec.shapes_by_dtype,
)
dtype_result = next(result for result in compile_results if result.dtype == dtype)
first_error = next(
    (result for result in compile_results if result.error_type is not None),
    None,
)

result = GenerationResult(
    source=decoded.source,
    model_id=MODEL_ID,
    grammar_active=True,
    kernel_class=spec.kernel_class,
    kernel_name=spec.name,
    dtype=dtype,
    compile_success=all(result.success for result in compile_results),
    compile_results_by_dtype={result.dtype: result.success for result in compile_results},
    compile_error_type=first_error.error_type if first_error is not None else None,
    compile_error_msg=first_error.error_msg if first_error is not None else None,
    masked_token_rate=decoded.masked_token_rate,
    unique_solution_hash=compute_unique_solution_hash(decoded.source),
    n_shapes_tested=dtype_result.n_shapes_tested,
    generation_seed=decoded.generation_seed,
    temperature=decoded.temperature,
    run_id=str(uuid4()),
    timestamp_utc=datetime.now(UTC).isoformat(),
)
validate_result_invariants(result)

print(json.dumps(asdict(result), indent=2))
